In [ ]:
#sharokku
from __future__ import annotations
import argparse
import logging
import os
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Subset

from src.dataset import AirfransGridDataset, PerChannelNormalizer
from src.losses import (
    compute_cl_cd,
    extract_freestream_speed,
    angle_of_attack_deg,
)
from src.models import GeoPINO

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("geo_pino.evaluate")

CH_NAMES = ["Ux  [m/s]", "Uy  [m/s]", "P  [Pa/ρ]", "nut  [m²/s]"]
CH_CMAPS = ["RdBu_r", "RdBu_r", "RdBu_r", "viridis"]


# Geometry-grouped split
def geometry_fingerprint(sdf_hw: np.ndarray, n_bins: int = 12) -> tuple:
    H, W = sdf_hw.shape
    sh, sw = max(H // n_bins, 1), max(W // n_bins, 1)
    small = sdf_hw[::sh, ::sw][:n_bins, :n_bins]
    return tuple(np.round(small, 2).ravel())


def build_geometry_groups(dataset: AirfransGridDataset) -> np.ndarray:
    fps, lookup = [], {}
    for i in range(len(dataset)):
        inp, _, _ = dataset[i]
        fp = geometry_fingerprint(inp[3].numpy())
        if fp not in lookup:
            lookup[fp] = len(lookup)
        fps.append(lookup[fp])
    return np.array(fps)


# Evaluation helpers
def evaluate_relL2(
    model: torch.nn.Module,
    loader: DataLoader,
    normaliser: PerChannelNormalizer,
    device: torch.device,
    label: str = "",
) -> Dict[str, float]:
    model.eval()
    accs = {nm: 0.0 for nm in ["Ux", "Uy", "P", "nut"]}
    n = 0
    with torch.no_grad():
        for vi, vt, vp in loader:
            vi, vt, vp = vi.to(device), vt.to(device), vp.to(device)
            pr, _ = model(vi, vp)
            pr_phys = normaliser.decode(pr)
            for k, nm in enumerate(["Ux", "Uy", "P", "nut"]):
                p_k = pr_phys[:, k].reshape(pr_phys.size(0), -1)
                t_k = vt[:, k].reshape(vt.size(0), -1)
                rel = ((p_k - t_k).norm(dim=1) /
                       t_k.norm(dim=1).clamp(min=1e-8)).mean()
                accs[nm] += float(rel)
            n += 1
    n = max(n, 1)
    log.info("[%s] %s", label,
             "  ".join(f"{k}={v/n*100:.2f}%" for k, v in accs.items()))
    return {k: v / n for k, v in accs.items()}


def collect_cl_cd(
    model: torch.nn.Module,
    loader: DataLoader,
    normaliser: PerChannelNormalizer,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    cl_true_list, cl_pred_list = [], []
    cd_true_list, cd_pred_list = [], []

    with torch.no_grad():
        for vi, vt, vp in loader:
            vi, vt, vp = vi.to(device), vt.to(device), vp.to(device)
            pr, _ = model(vi, vp)
            pr_phys = normaliser.decode(pr)
            sdf = vi[:, 3:4]
            v_inf = extract_freestream_speed(vi)

            t_cl, t_cd = compute_cl_cd(vt, sdf, vp, v_inf)
            p_cl, p_cd = compute_cl_cd(pr_phys, sdf, vp, v_inf)

            cl_true_list.extend(t_cl.cpu().tolist())
            cl_pred_list.extend(p_cl.cpu().tolist())
            cd_true_list.extend(t_cd.cpu().tolist())
            cd_pred_list.extend(p_cd.cpu().tolist())

    return (
        np.array(cl_true_list), np.array(cl_pred_list),
        np.array(cd_true_list), np.array(cd_pred_list),
    )


# Plotting
def _add_colorbar(ax, im, label: str = "") -> None:
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size="4%", pad=0.05)
    plt.colorbar(im, cax=cax, label=label)


def plot_aero_coeffs(
    cl_true: np.ndarray, cl_pred: np.ndarray,
    cd_true: np.ndarray, cd_pred: np.ndarray,
    out_path: str,
) -> None:
    r2_cl = r2_score(cl_true, cl_pred)
    r2_cd = r2_score(cd_true, cd_pred)
    sp_cl, _ = spearmanr(cl_true, cl_pred)
    sp_cd, _ = spearmanr(cd_true, cd_pred)
    mae_cl = float(np.mean(np.abs(cl_true - cl_pred)))
    mae_cd = float(np.mean(np.abs(cd_true - cd_pred)))

    log.info("Cl — R²=%.4f  Spearman=%.4f  MAE=%.4f", r2_cl, sp_cl, mae_cl)
    log.info("Cd — R²=%.4f  Spearman=%.4f  MAE=%.4f", r2_cd, sp_cd, mae_cd)

    fig, axs = plt.subplots(1, 2, figsize=(14, 6), facecolor="#f8f9fa")
    fig.suptitle(
        "Aerodynamic Coefficients: Geo-PINO vs. OpenFOAM (OOD Test Set)",
        fontsize=15, fontweight="bold",
    )

    specs = [
        (axs[0], cl_true, cl_pred, r"Lift Coefficient ($C_L$)",
         "#0ea5e9", r2_cl, sp_cl, mae_cl),
        (axs[1], cd_true, cd_pred, r"Drag Coefficient ($C_D$)",
         "#ef4444", r2_cd, sp_cd, mae_cd),
    ]
    for ax, t_vals, p_vals, name, col, r2, sp, mae in specs:
        ax.scatter(t_vals, p_vals, alpha=0.75, color=col,
                   edgecolors="white", s=70, zorder=3)
        lo = min(t_vals.min(), p_vals.min())
        hi = max(t_vals.max(), p_vals.max())
        pad = (hi - lo) * 0.1
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
                "k--", lw=1.5, label="Ideal ($y = x$)", zorder=2)
        ax.set_title(name, fontsize=13, fontweight="semibold")
        ax.set_xlabel("OpenFOAM (True CFD)", fontsize=11)
        ax.set_ylabel("Geo-PINO (Predicted)", fontsize=11)
        ax.set_xlim([lo - pad, hi + pad])
        ax.set_ylim([lo - pad, hi + pad])
        ax.grid(True, linestyle="--", linewidth=0.5, color="#cbd5e1", zorder=1)
        ax.legend(loc="upper left")
        stats_text = f"$R^2$ = {r2:.4f}\nSpearman = {sp:.4f}\nMAE = {mae:.4f}"
        ax.text(0.95, 0.05, stats_text,
                transform=ax.transAxes, fontsize=11,
                va="bottom", ha="right",
                bbox=dict(boxstyle="round,pad=0.5",
                          facecolor="white", alpha=0.9, edgecolor="#cbd5e1"))

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info("Aerodynamic coefficients plot → %s", out_path)


def plot_error_maps(
    model: torch.nn.Module,
    loader: DataLoader,
    normaliser: PerChannelNormalizer,
    device: torch.device,
    out_path: str,
) -> None:
    model.eval()
    with torch.no_grad():
        si, st, sp = next(iter(loader))
        pr, _ = model(si[:1].to(device), sp[:1].to(device))
        pred_phys = normaliser.decode(pr).cpu().numpy()[0]   # [4, H, W]
        true_phys = st[0].numpy()                             # [4, H, W]
        mask_np = si[0, 0].numpy()                            # [H, W]

    fig, axs = plt.subplots(1, 2, figsize=(16, 6), facecolor="#f8f9fa")
    fig.suptitle(
        "Absolute Error Fields - Geo-PINO vs. OpenFOAM (OOD Sample)",
        fontsize=15, fontweight="bold",
    )
    panels = [
        {"idx": 0, "name": r"Velocity $U_x$ Absolute Error [m/s]"},
        {"idx": 2, "name": r"Pressure $P$ Absolute Error [Pa/ρ]"},
    ]
    for i, ch in enumerate(panels):
        ax = axs[i]
        err = np.abs(true_phys[ch["idx"]] - pred_phys[ch["idx"]])
        err_masked = np.ma.masked_where(mask_np == 1, err)

        cmap = plt.get_cmap("magma_r").copy()
        cmap.set_bad(color="#1e293b")

        vmax = float(np.percentile(err_masked.compressed(), 99.5))
        im = ax.imshow(err_masked.T, origin="lower", aspect="auto",
                       cmap=cmap, vmin=0, vmax=vmax)
        ax.set_title(ch["name"], fontsize=13, fontweight="semibold", pad=10)
        ax.axis("off")
        _add_colorbar(ax, im)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info("Error maps → %s", out_path)


def plot_pred_vs_gt(
    model: torch.nn.Module,
    loader: DataLoader,
    normaliser: PerChannelNormalizer,
    device: torch.device,
    out_path: str,
) -> None:
    import matplotlib.gridspec as gridspec

    model.eval()
    with torch.no_grad():
        si, st, sp = next(iter(loader))
        pr, _ = model(si[:1].to(device), sp[:1].to(device))
        pred_phys = normaliser.decode(pr).cpu().numpy()[0]
        true_phys = st[0].numpy()
        mask_np = si[0, 0].numpy()

    fig = plt.figure(figsize=(20, 9))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle("Geo-PINO — Prediction vs Ground Truth", fontsize=14, y=1.01)

    for k in range(4):
        p_ch = np.ma.masked_where(mask_np == 1, pred_phys[k])
        t_ch = np.ma.masked_where(mask_np == 1, true_phys[k])
        vmin = float(min(p_ch.min(), t_ch.min()))
        vmax = float(max(p_ch.max(), t_ch.max()))

        ax = fig.add_subplot(gs[0, k])
        im = ax.imshow(p_ch.T, origin="lower", aspect="auto",
                       cmap=CH_CMAPS[k], vmin=vmin, vmax=vmax)
        ax.set_title(f"Pred  {CH_NAMES[k]}", fontsize=9)
        ax.axis("off")
        _add_colorbar(ax, im)

        ax = fig.add_subplot(gs[1, k])
        im = ax.imshow(t_ch.T, origin="lower", aspect="auto",
                       cmap=CH_CMAPS[k], vmin=vmin, vmax=vmax)
        ax.set_title(f"True  {CH_NAMES[k]}", fontsize=9)
        ax.axis("off")
        _add_colorbar(ax, im)

        err = np.abs(pred_phys[k] - true_phys[k])
        err_m = np.ma.masked_where(mask_np == 1, err)
        rel = float(np.linalg.norm(err_m.compressed()) /
                    (np.linalg.norm(t_ch.compressed()) + 1e-8))
        ax = fig.add_subplot(gs[2, k])
        im = ax.imshow(err_m.T, origin="lower", aspect="auto",
                       cmap="hot_r", vmin=0)
        ax.set_title(f"|Error|  relL2={rel:.3f}", fontsize=9)
        ax.axis("off")
        _add_colorbar(ax, im)

    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info("Pred vs GT → %s", out_path)


# Argument parsing
def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        "geo_pino_evaluate",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--checkpoint", type=str, required=True,
                   help="Path to the .pt checkpoint produced by train.py")
    p.add_argument("--data_dir", type=str, default="./data/airfrans_prep")
    p.add_argument("--output_dir", type=str, default="./output/eval")
    p.add_argument("--samples", type=int, default=None,
                   help="Maximum number of samples to load (None = all)")
    p.add_argument("--grid_size", type=int, default=241)
    p.add_argument("--batch_size", type=int, default=4)
    p.add_argument("--ood_aoa_pct", type=float, default=80.0,
                   help="Percentile AoA threshold: samples above go to OOD test set")
    p.add_argument("--num_workers", type=int, default=0)
    p.add_argument("--seed", type=int, default=0)
    return p.parse_args(argv)


# Main
def main() -> None:
    args = parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.output_dir, exist_ok=True)
    log.info("Device: %s | Checkpoint: %s", device, args.checkpoint)

    #Load checkpoint
    ckpt = torch.load(args.checkpoint, map_location=device, weights_only=False)
    train_args = ckpt.get("args", {})

    model = GeoPINO(
        in_ch=4, out_ch=4,
        modes=train_args.get("modes", 20),
        width=train_args.get("width", 64),
        n_layers=train_args.get("n_layers", 4),
        dropout_p=train_args.get("dropout", 0.1),
        use_iphi=not train_args.get("no_iphi", False),
    ).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    log.info("Model loaded from checkpoint")

    # Reconstruct normaliser from saved statistics
    norm = object.__new__(PerChannelNormalizer)
    norm.mean = ckpt["norm_mean"].to(device)
    norm.std = ckpt["norm_std"].to(device)
    norm.clip = 5.0

    #Full dataset
    log.info("[1/5] Dataset indexing")
    cache_dir = os.path.join(args.output_dir, f"bary_cache_{args.grid_size}")
    dset = AirfransGridDataset(
        root_dir=args.data_dir,
        grid_size=args.grid_size,
        num_samples=args.samples,
        cache_dir=cache_dir,
        allow_synthetic=False,
    )

    #Geometry-grouped train/val split (no shape leakage)
    log.info("[2/5] Geometry-grouped split")
    groups = build_geometry_groups(dset)
    n_groups = len(set(groups))
    log.info("%d samples → %d distinct geometry groups", len(dset), n_groups)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2,
                            random_state=args.seed)
    train_idx, val_idx = next(gss.split(np.zeros(len(dset)), groups=groups))
    overlap = set(groups[train_idx]) & set(groups[val_idx])
    log.info("Geometry-grouped split: train=%d  val=%d  shape-overlap=%d (should be 0)",
             len(train_idx), len(val_idx), len(overlap))

    va_ds = Subset(dset, val_idx)
    va_ld = DataLoader(va_ds, batch_size=args.batch_size, shuffle=False,
                       num_workers=args.num_workers)

    #AoA-extrapolation OOD split
    log.info("[3/5] AoA-extrapolation OOD split")
    aoas = np.array([angle_of_attack_deg(dset[i][0]) for i in range(len(dset))])
    log.info("AoA range: [%.2f°, %.2f°]", aoas.min(), aoas.max())

    ood_thresh = float(np.percentile(aoas, args.ood_aoa_pct))
    id_idx = np.where(aoas < ood_thresh)[0]
    ood_idx = np.where(aoas >= ood_thresh)[0]
    log.info("In-distribution:  %d samples  (AoA < %.2f°)", len(id_idx), ood_thresh)
    log.info("OOD test set:     %d samples  (AoA ≥ %.2f°)", len(ood_idx), ood_thresh)

    ood_ds = Subset(dset, ood_idx)
    ood_ld = DataLoader(ood_ds, batch_size=args.batch_size, shuffle=False,
                        num_workers=args.num_workers)

    #Per-channel RelL2 metrics
    log.info("[4/5] Per-channel RelL2 metrics")
    log.info("--- Geometry-grouped validation ---")
    val_metrics = evaluate_relL2(model, va_ld, norm, device,
                                 label="Geometry-grouped val")
    log.info("--- OOD test (AoA extrapolation) ---")
    ood_metrics = evaluate_relL2(model, ood_ld, norm, device,
                                 label="OOD test")

    # Aerodynamic coefficients + Spearman
    log.info("[5/5] Aerodynamic coefficients (OOD)")
    cl_true, cl_pred, cd_true, cd_pred = collect_cl_cd(
        model, ood_ld, norm, device)

    r2_cl = r2_score(cl_true, cl_pred)
    r2_cd = r2_score(cd_true, cd_pred)
    sp_cl, _ = spearmanr(cl_true, cl_pred)
    sp_cd, _ = spearmanr(cd_true, cd_pred)

    log.info("Sanity check — True Cl range: [%.4f, %.4f]  (expect -0.5 .. 1.8)",
             cl_true.min(), cl_true.max())
    log.info("Sanity check — True Cd range: [%.4f, %.4f]  (expect 0.005 .. 0.3)",
             cd_true.min(), cd_true.max())

    #Save figures
    plot_aero_coeffs(
        cl_true, cl_pred, cd_true, cd_pred,
        out_path=os.path.join(args.output_dir, "aero_coeffs.png"),
    )
    plot_error_maps(
        model, ood_ld, norm, device,
        out_path=os.path.join(args.output_dir, "error_maps.png"),
    )
    plot_pred_vs_gt(
        model, va_ld, norm, device,
        out_path=os.path.join(args.output_dir, "pred_vs_gt.png"),
    )

    #Final summary
    log.info("━" * 64)
    log.info("Geo-PINO - Evaluation Summary (OOD test set)")
    log.info("  Per-channel RelL2:")
    for k, v in ood_metrics.items():
        log.info("    %-4s  %.2f%%", k, v * 100)
    log.info("  Aerodynamic coefficients (OOD):")
    log.info("    Cl  R²=%.4f  Spearman=%.4f", r2_cl, sp_cl)
    log.info("    Cd  R²=%.4f  Spearman=%.4f", r2_cd, sp_cd)
    log.info("  Outputs saved to: %s", args.output_dir)
    log.info("━" * 64)


if __name__ == "__main__":
    main()